# Advanced Problems with Solutions — Custom JSON Decoding in Python

This notebook builds on `json.loads`, `object_hook`, `object_pairs_hook`, `parse_float`, `parse_int`, `parse_constant`, and `JSONDecoder`.

## Goals

By the end, you should be able to:

- Decode tagged JSON objects into rich Python types.
- Understand the **bottom-up** behavior of `object_hook`.
- Decode exact numeric values with `Decimal`.
- Reject non-standard JSON constants such as `NaN` and `Infinity`.
- Detect duplicate object keys with `object_pairs_hook`.
- Build a registry-based decoder that scales better than a long `if/elif` chain.
- Decode dataclasses and nested domain models safely.
- Handle schema versions and migrations.
- Subclass `json.JSONDecoder`.
- Design strict, secure decoders that do **not** instantiate arbitrary classes.
- Test custom decoders with assertions and round-trip checks.

> Best practice: treat JSON decoding as **data interpretation**, not merely parsing. JSON itself only knows objects, arrays, strings, numbers, booleans, and null.


## Setup


In [1]:
import json
from dataclasses import dataclass, asdict
from datetime import datetime, date, timezone
from decimal import Decimal
from fractions import Fraction
from pathlib import Path
from typing import Any, Callable
from uuid import UUID


## A small testing helper

The problems below use plain `assert` statements so the notebook requires no third-party test framework.


In [2]:
def assert_type(value, expected_type):
    assert isinstance(value, expected_type), (
        f"Expected {expected_type.__name__}, got {type(value).__name__}: {value!r}"
    )

def assert_equal(actual, expected):
    assert actual == expected, f"Expected {expected!r}, got {actual!r}"

print("Helpers ready.")


Helpers ready.


---

# Problem 1 — Decode Multiple Tagged Types at Any Nesting Depth

You receive JSON objects using the following tagged schemas:

```json
{"objecttype": "datetime", "value": "2026-08-07T12:30:45"}
{"objecttype": "fraction", "numerator": 3, "denominator": 7}
```

Create one `object_hook` that:

1. Converts tagged datetimes into `datetime`.
2. Converts tagged fractions into `Fraction`.
3. Leaves unrecognized dictionaries unchanged.
4. Works automatically for nested structures.

### Input


In [3]:
problem_1_json = r'''
{
  "job": {
    "started": {
      "objecttype": "datetime",
      "value": "2026-08-07T12:30:45"
    },
    "ratio": {
      "objecttype": "fraction",
      "numerator": 3,
      "denominator": 7
    }
  },
  "status": "running"
}
'''


## Solution 1


In [4]:
def basic_tagged_decoder(obj):
    tag = obj.get("objecttype")

    if tag == "datetime":
        return datetime.strptime(obj["value"], "%Y-%m-%dT%H:%M:%S")

    if tag == "fraction":
        return Fraction(obj["numerator"], obj["denominator"])

    return obj

decoded_1 = json.loads(problem_1_json, object_hook=basic_tagged_decoder)
decoded_1


{'job': {'started': datetime.datetime(2026, 8, 7, 12, 30, 45),
  'ratio': Fraction(3, 7)},
 'status': 'running'}

In [5]:
assert_type(decoded_1["job"]["started"], datetime)
assert_equal(decoded_1["job"]["ratio"], Fraction(3, 7))
assert_equal(decoded_1["status"], "running")

print("Problem 1 passed.")


Problem 1 passed.


### Why this works

`object_hook` is called for every decoded JSON object, starting with the deepest objects first. By the time the parent dictionary is passed to the hook, its tagged child dictionaries may already have become Python objects.


---

# Problem 2 — Prove That `object_hook` Runs Bottom-Up

Create a diagnostic hook that records the order in which dictionaries are processed.

Use this JSON:

```json
{
  "outer": {
    "middle": {
      "inner": {"x": 1}
    }
  }
}
```

Your observed order should show `inner` before `middle`, and `middle` before the root.


## Solution 2


In [6]:
problem_2_json = r'''
{
  "outer": {
    "middle": {
      "inner": {"x": 1}
    }
  }
}
'''

hook_calls = []

def tracing_hook(obj):
    hook_calls.append(dict(obj))
    return obj

decoded_2 = json.loads(problem_2_json, object_hook=tracing_hook)

for index, item in enumerate(hook_calls, start=1):
    print(index, item)


1 {'x': 1}
2 {'inner': {'x': 1}}
3 {'middle': {'inner': {'x': 1}}}
4 {'outer': {'middle': {'inner': {'x': 1}}}}


In [7]:
assert len(hook_calls) == 4
assert hook_calls[0] == {"x": 1}
assert "inner" in hook_calls[1]
assert "middle" in hook_calls[2]
assert "outer" in hook_calls[3]

print("Problem 2 passed.")


Problem 2 passed.


### Best-practice takeaway

Bottom-up decoding is extremely useful for nested domain models. A parent decoder can receive already-decoded children.


---

# Problem 3 — Decode ISO 8601 Datetimes, Including Time Zones

The simple format `%Y-%m-%dT%H:%M:%S` does not handle offsets such as `+03:00` or the UTC suffix `Z`.

Implement a decoder that accepts:

- `2026-08-07T10:15:00`
- `2026-08-07T10:15:00+03:00`
- `2026-08-07T07:15:00Z`

Use `datetime.fromisoformat`.


## Solution 3


In [8]:
def parse_iso_datetime(value: str) -> datetime:
    # Python's fromisoformat accepts "+00:00"; normalize the common "Z" form.
    normalized = value[:-1] + "+00:00" if value.endswith("Z") else value
    return datetime.fromisoformat(normalized)

def iso_datetime_decoder(obj):
    if obj.get("objecttype") == "datetime":
        return parse_iso_datetime(obj["value"])
    return obj

problem_3_json = r'''
{
  "local": {
    "objecttype": "datetime",
    "value": "2026-08-07T10:15:00"
  },
  "sofia": {
    "objecttype": "datetime",
    "value": "2026-08-07T10:15:00+03:00"
  },
  "utc": {
    "objecttype": "datetime",
    "value": "2026-08-07T07:15:00Z"
  }
}
'''

decoded_3 = json.loads(problem_3_json, object_hook=iso_datetime_decoder)
decoded_3


{'local': datetime.datetime(2026, 8, 7, 10, 15),
 'sofia': datetime.datetime(2026, 8, 7, 10, 15, tzinfo=datetime.timezone(datetime.timedelta(seconds=10800))),
 'utc': datetime.datetime(2026, 8, 7, 7, 15, tzinfo=datetime.timezone.utc)}

In [9]:
assert decoded_3["local"].tzinfo is None
assert decoded_3["sofia"].utcoffset().total_seconds() == 3 * 3600
assert decoded_3["utc"].utcoffset().total_seconds() == 0

print("Problem 3 passed.")


Problem 3 passed.


### Best practice

Prefer ISO 8601-compatible text for dates and datetimes. For new code, `datetime.fromisoformat` is usually more flexible and readable than a single hard-coded `strptime` format.


---

# Problem 4 — Decode Monetary Numbers Exactly with `Decimal`

Binary floating point can produce surprising results for values such as `0.1`.

Parse every JSON floating-point literal as a `Decimal` by using `parse_float=Decimal`.

Then compute:

```python
0.1 + 0.2
```

from decoded JSON and prove that the result is exactly `Decimal("0.3")`.


## Solution 4


In [10]:
problem_4_json = r'''
{
  "price": 0.1,
  "tax": 0.2,
  "quantity": 3
}
'''

decoded_4 = json.loads(problem_4_json, parse_float=Decimal)
decoded_4


{'price': Decimal('0.1'), 'tax': Decimal('0.2'), 'quantity': 3}

In [11]:
total = decoded_4["price"] + decoded_4["tax"]

assert_type(decoded_4["price"], Decimal)
assert_type(decoded_4["tax"], Decimal)
assert_type(decoded_4["quantity"], int)
assert_equal(total, Decimal("0.3"))

print("Exact total:", total)
print("Problem 4 passed.")


Exact total: 0.3
Problem 4 passed.


### Important detail

`parse_float` receives the numeric token as a **string**, which allows `Decimal` to construct the exact decimal value without first going through a binary `float`.


---

# Problem 5 — Apply a Custom Integer Policy with `parse_int`

Suppose an API must reject integer values outside the signed 64-bit range:

- minimum: `-2**63`
- maximum: `2**63 - 1`

Write a custom `parse_int` callable that validates every JSON integer before returning it.


## Solution 5


In [12]:
INT64_MIN = -(2**63)
INT64_MAX = 2**63 - 1

def parse_int64(token: str) -> int:
    value = int(token)
    if not INT64_MIN <= value <= INT64_MAX:
        raise ValueError(f"Integer outside signed 64-bit range: {token}")
    return value

valid_int_json = '{"small": 42, "large": 9223372036854775807}'
decoded_5 = json.loads(valid_int_json, parse_int=parse_int64)

assert_equal(decoded_5["small"], 42)
assert_equal(decoded_5["large"], INT64_MAX)

print(decoded_5)


{'small': 42, 'large': 9223372036854775807}


In [13]:
too_large_json = '{"value": 9223372036854775808}'

try:
    json.loads(too_large_json, parse_int=parse_int64)
except ValueError as exc:
    print("Rejected as expected:", exc)
else:
    raise AssertionError("Expected the oversized integer to be rejected.")

print("Problem 5 passed.")


Rejected as expected: Integer outside signed 64-bit range: 9223372036854775808
Problem 5 passed.


---

# Problem 6 — Reject `NaN`, `Infinity`, and `-Infinity`

Python's `json` module accepts the non-standard constants `NaN`, `Infinity`, and `-Infinity` by default.

For strict interchange, reject them with `parse_constant`.


## Solution 6


In [14]:
def reject_nonstandard_constant(token: str):
    raise ValueError(f"Non-standard JSON constant is not allowed: {token}")

strict_json = r'''
{
  "temperature": 21.5,
  "reading": NaN
}
'''

try:
    json.loads(strict_json, parse_constant=reject_nonstandard_constant)
except ValueError as exc:
    print("Rejected as expected:", exc)
else:
    raise AssertionError("Expected NaN to be rejected.")

print("Problem 6 passed.")


Rejected as expected: Non-standard JSON constant is not allowed: NaN
Problem 6 passed.


### Best practice

When consuming external data that must follow the JSON standard strictly, explicitly reject these constants. Silent acceptance can hide invalid upstream data.


---

# Problem 7 — Detect Duplicate Object Keys

A JSON object such as:

```json
{"role": "user", "role": "admin"}
```

contains a duplicate key. A normal dictionary cannot preserve both occurrences, and the later value usually wins.

Use `object_pairs_hook` to detect duplicates and raise an error.


## Solution 7


In [15]:
def reject_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key!r}")
        result[key] = value
    return result

safe_json = '{"role": "user", "active": true}'
assert_equal(
    json.loads(safe_json, object_pairs_hook=reject_duplicate_keys),
    {"role": "user", "active": True},
)

duplicate_json = '{"role": "user", "role": "admin"}'

try:
    json.loads(duplicate_json, object_pairs_hook=reject_duplicate_keys)
except ValueError as exc:
    print("Rejected duplicate:", exc)
else:
    raise AssertionError("Expected duplicate key rejection.")

print("Problem 7 passed.")


Rejected duplicate: Duplicate JSON key: 'role'
Problem 7 passed.


### Why this matters

Duplicate keys can be a correctness and security problem because different parsers may resolve them differently.


---

# Problem 8 — Understand Hook Precedence

What happens when both `object_hook` and `object_pairs_hook` are supplied?

Demonstrate that `object_pairs_hook` takes precedence.


## Solution 8


In [16]:
events = []

def demo_object_hook(obj):
    events.append(("object_hook", obj))
    return obj

def demo_pairs_hook(pairs):
    events.append(("object_pairs_hook", pairs))
    return dict(pairs)

decoded_8 = json.loads(
    '{"a": {"b": 1}}',
    object_hook=demo_object_hook,
    object_pairs_hook=demo_pairs_hook,
)

print("Decoded:", decoded_8)
print("Events:", events)


Decoded: {'a': {'b': 1}}
Events: [('object_pairs_hook', [('b', 1)]), ('object_pairs_hook', [('a', {'b': 1})])]


In [17]:
assert events
assert all(name == "object_pairs_hook" for name, _ in events)

print("Problem 8 passed.")


Problem 8 passed.


### Best practice

Do not pass both unless you deliberately want `object_pairs_hook` to win. If you need duplicate-key handling **and** custom object decoding, combine those responsibilities in one `object_pairs_hook`, as shown later.


---

# Problem 9 — Avoid Accidental Tag Collisions

This object is normal business data:

```json
{"objecttype": "datetime", "value": "not actually a datetime", "label": "user data"}
```

A naive decoder may attempt to convert it merely because `objecttype == "datetime"`.

Design a stricter tagged schema using a reserved marker:

```json
{"__type__": "datetime", "__version__": 1, "value": "..."}
```

Validate the expected keys before decoding.


## Solution 9


In [18]:
def strict_tagged_decoder(obj):
    tag = obj.get("__type__")
    if tag is None:
        return obj

    version = obj.get("__version__")
    if version != 1:
        raise ValueError(f"Unsupported tagged-object version: {version!r}")

    if tag == "datetime":
        expected = {"__type__", "__version__", "value"}
        if set(obj) != expected:
            raise ValueError(
                f"Invalid datetime object keys: expected {expected}, got {set(obj)}"
            )
        return parse_iso_datetime(obj["value"])

    return obj

normal_business_data = r'''
{
  "objecttype": "datetime",
  "value": "not actually a datetime",
  "label": "user data"
}
'''

decoded_normal = json.loads(normal_business_data, object_hook=strict_tagged_decoder)
assert_type(decoded_normal, dict)
assert_equal(decoded_normal["objecttype"], "datetime")

tagged_datetime = r'''
{
  "__type__": "datetime",
  "__version__": 1,
  "value": "2026-08-07T16:00:00+03:00"
}
'''

decoded_tagged = json.loads(tagged_datetime, object_hook=strict_tagged_decoder)
assert_type(decoded_tagged, datetime)

print("Problem 9 passed.")


Problem 9 passed.


---

# Problem 10 — Replace a Long `if/elif` Chain with a Decoder Registry

As supported types grow, a long conditional decoder becomes difficult to maintain.

Build a registry mapping type tags to decoder functions.

Support:

- `datetime`
- `fraction`
- `uuid`
- `date`


## Solution 10


In [19]:
DecoderFunction = Callable[[dict], Any]

def decode_datetime_record(obj):
    return parse_iso_datetime(obj["value"])

def decode_fraction_record(obj):
    return Fraction(obj["numerator"], obj["denominator"])

def decode_uuid_record(obj):
    return UUID(obj["value"])

def decode_date_record(obj):
    return date.fromisoformat(obj["value"])

DECODER_REGISTRY: dict[str, DecoderFunction] = {
    "datetime": decode_datetime_record,
    "fraction": decode_fraction_record,
    "uuid": decode_uuid_record,
    "date": decode_date_record,
}

def registry_decoder(obj):
    tag = obj.get("__type__")
    if tag is None:
        return obj

    decoder = DECODER_REGISTRY.get(tag)
    if decoder is None:
        # Unknown tags are intentionally left as raw dictionaries here.
        return obj

    return decoder(obj)


In [20]:
problem_10_json = r'''
{
  "created": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00Z"
  },
  "birthday": {
    "__type__": "date",
    "value": "1995-04-23"
  },
  "ratio": {
    "__type__": "fraction",
    "numerator": 5,
    "denominator": 9
  },
  "request_id": {
    "__type__": "uuid",
    "value": "12345678-1234-5678-1234-567812345678"
  }
}
'''

decoded_10 = json.loads(problem_10_json, object_hook=registry_decoder)
decoded_10


{'created': datetime.datetime(2026, 8, 7, 12, 0, tzinfo=datetime.timezone.utc),
 'birthday': datetime.date(1995, 4, 23),
 'ratio': Fraction(5, 9),
 'request_id': UUID('12345678-1234-5678-1234-567812345678')}

In [21]:
assert_type(decoded_10["created"], datetime)
assert_type(decoded_10["birthday"], date)
assert_equal(decoded_10["ratio"], Fraction(5, 9))
assert_equal(
    decoded_10["request_id"],
    UUID("12345678-1234-5678-1234-567812345678"),
)

print("Problem 10 passed.")


Problem 10 passed.


### Best-practice advantage

The registry separates **dispatch** from **conversion logic**, making it easier to test, extend, and review each decoder independently.


---

# Problem 11 — Implement Strict Unknown-Type Handling

For an internal protocol, silently preserving an unknown tagged type is dangerous because it may indicate a version mismatch.

Create a strict registry decoder that:

- leaves untagged dictionaries alone;
- decodes known tags;
- raises an error for unknown tags.


## Solution 11


In [22]:
class UnknownTaggedTypeError(ValueError):
    pass

def strict_registry_decoder(obj):
    tag = obj.get("__type__")
    if tag is None:
        return obj

    try:
        decoder = DECODER_REGISTRY[tag]
    except KeyError as exc:
        raise UnknownTaggedTypeError(f"Unsupported tagged type: {tag!r}") from exc

    return decoder(obj)

unknown_json = r'''
{
  "payload": {
    "__type__": "complex-number",
    "real": 1,
    "imag": 2
  }
}
'''

try:
    json.loads(unknown_json, object_hook=strict_registry_decoder)
except UnknownTaggedTypeError as exc:
    print("Rejected as expected:", exc)
else:
    raise AssertionError("Expected unknown type rejection.")

print("Problem 11 passed.")


Rejected as expected: Unsupported tagged type: 'complex-number'
Problem 11 passed.


---

# Problem 12 — Decode a Dataclass

Create a `Person` dataclass and decode this schema:

```json
{
  "__type__": "person",
  "name": "Ada Lovelace",
  "employee_id": 1001
}
```

Validate fields before constructing the object.


## Solution 12


In [23]:
@dataclass(frozen=True)
class Person:
    name: str
    employee_id: int

def decode_person_record(obj):
    expected = {"__type__", "name", "employee_id"}

    if set(obj) != expected:
        missing = expected - set(obj)
        extra = set(obj) - expected
        raise ValueError(
            f"Invalid person record. Missing={sorted(missing)}, extra={sorted(extra)}"
        )

    if not isinstance(obj["name"], str):
        raise TypeError("Person.name must be a string.")

    if not isinstance(obj["employee_id"], int):
        raise TypeError("Person.employee_id must be an integer.")

    return Person(
        name=obj["name"],
        employee_id=obj["employee_id"],
    )

DECODER_REGISTRY_WITH_PERSON = {
    **DECODER_REGISTRY,
    "person": decode_person_record,
}

def person_registry_decoder(obj):
    tag = obj.get("__type__")
    if tag is None:
        return obj

    decoder = DECODER_REGISTRY_WITH_PERSON.get(tag)
    if decoder is None:
        raise UnknownTaggedTypeError(tag)

    return decoder(obj)

person_json = r'''
{
  "owner": {
    "__type__": "person",
    "name": "Ada Lovelace",
    "employee_id": 1001
  }
}
'''

decoded_12 = json.loads(person_json, object_hook=person_registry_decoder)
decoded_12


{'owner': Person(name='Ada Lovelace', employee_id=1001)}

In [24]:
assert_equal(decoded_12["owner"], Person("Ada Lovelace", 1001))
print("Problem 12 passed.")


Problem 12 passed.


---

# Problem 13 — Decode Nested Domain Objects Bottom-Up

Define:

- `LineItem(sku, quantity, unit_price)`
- `Order(order_id, customer, items, created_at)`

The JSON should tag each line item and the order itself.

Requirements:

1. Decode monetary values with `Decimal`.
2. Decode `created_at` as a tagged datetime.
3. Decode each tagged line item into `LineItem`.
4. Decode the outer order into `Order`.
5. Exploit bottom-up `object_hook` behavior.


## Solution 13


In [25]:
@dataclass(frozen=True)
class LineItem:
    sku: str
    quantity: int
    unit_price: Decimal

    @property
    def subtotal(self) -> Decimal:
        return self.unit_price * self.quantity

@dataclass(frozen=True)
class Order:
    order_id: str
    customer: Person
    items: tuple[LineItem, ...]
    created_at: datetime

    @property
    def total(self) -> Decimal:
        return sum((item.subtotal for item in self.items), Decimal("0"))


In [26]:
def decode_line_item(obj):
    return LineItem(
        sku=obj["sku"],
        quantity=obj["quantity"],
        unit_price=obj["unit_price"],
    )

def decode_order(obj):
    customer = obj["customer"]
    items = obj["items"]
    created_at = obj["created_at"]

    if not isinstance(customer, Person):
        raise TypeError("Order.customer was not decoded to Person.")

    if not all(isinstance(item, LineItem) for item in items):
        raise TypeError("Every Order.items entry must be a LineItem.")

    if not isinstance(created_at, datetime):
        raise TypeError("Order.created_at was not decoded to datetime.")

    return Order(
        order_id=obj["order_id"],
        customer=customer,
        items=tuple(items),
        created_at=created_at,
    )

DOMAIN_REGISTRY = {
    **DECODER_REGISTRY_WITH_PERSON,
    "line_item": decode_line_item,
    "order": decode_order,
}

def domain_decoder(obj):
    tag = obj.get("__type__")
    if tag is None:
        return obj

    decoder = DOMAIN_REGISTRY.get(tag)
    if decoder is None:
        raise UnknownTaggedTypeError(f"Unknown domain type: {tag!r}")

    return decoder(obj)


In [27]:
order_json = r'''
{
  "__type__": "order",
  "order_id": "ORD-2026-0001",
  "customer": {
    "__type__": "person",
    "name": "Grace Hopper",
    "employee_id": 7
  },
  "items": [
    {
      "__type__": "line_item",
      "sku": "PY-BOOK",
      "quantity": 2,
      "unit_price": 19.95
    },
    {
      "__type__": "line_item",
      "sku": "JSON-MUG",
      "quantity": 1,
      "unit_price": 8.50
    }
  ],
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T09:30:00Z"
  }
}
'''

decoded_13 = json.loads(
    order_json,
    object_hook=domain_decoder,
    parse_float=Decimal,
)

decoded_13


Order(order_id='ORD-2026-0001', customer=Person(name='Grace Hopper', employee_id=7), items=(LineItem(sku='PY-BOOK', quantity=2, unit_price=Decimal('19.95')), LineItem(sku='JSON-MUG', quantity=1, unit_price=Decimal('8.50'))), created_at=datetime.datetime(2026, 8, 7, 9, 30, tzinfo=datetime.timezone.utc))

In [28]:
assert_type(decoded_13, Order)
assert_type(decoded_13.customer, Person)
assert all(isinstance(item, LineItem) for item in decoded_13.items)
assert_type(decoded_13.created_at, datetime)
assert_equal(decoded_13.total, Decimal("48.40"))

print("Order total:", decoded_13.total)
print("Problem 13 passed.")


Order total: 48.40
Problem 13 passed.


### Key insight

The `order` decoder can safely expect its nested tagged dictionaries to have been converted already. That is one of the most powerful properties of `object_hook`.


---

# Problem 14 — Combine Duplicate-Key Detection with Tagged Decoding

Because `object_pairs_hook` takes precedence over `object_hook`, create one `object_pairs_hook` that:

1. rejects duplicate keys;
2. converts the pair list to a dictionary;
3. applies the domain decoder.

This gives you both strict key handling and custom object reconstruction.


## Solution 14


In [29]:
def strict_domain_pairs_hook(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate key not allowed: {key!r}")
        obj[key] = value

    return domain_decoder(obj)

decoded_14 = json.loads(
    order_json,
    object_pairs_hook=strict_domain_pairs_hook,
    parse_float=Decimal,
)

assert_type(decoded_14, Order)
assert_equal(decoded_14.total, Decimal("48.40"))

print(decoded_14)
print("Problem 14 passed.")


Order(order_id='ORD-2026-0001', customer=Person(name='Grace Hopper', employee_id=7), items=(LineItem(sku='PY-BOOK', quantity=2, unit_price=Decimal('19.95')), LineItem(sku='JSON-MUG', quantity=1, unit_price=Decimal('8.50'))), created_at=datetime.datetime(2026, 8, 7, 9, 30, tzinfo=datetime.timezone.utc))
Problem 14 passed.


In [30]:
bad_order_json = r'''
{
  "__type__": "order",
  "order_id": "A",
  "order_id": "B",
  "customer": {
    "__type__": "person",
    "name": "Duplicate Test",
    "employee_id": 1
  },
  "items": [],
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T00:00:00Z"
  }
}
'''

try:
    json.loads(
        bad_order_json,
        object_pairs_hook=strict_domain_pairs_hook,
        parse_float=Decimal,
    )
except ValueError as exc:
    print("Duplicate rejected:", exc)
else:
    raise AssertionError("Expected duplicate key rejection.")

print("Problem 14 duplicate-key test passed.")


Duplicate rejected: Duplicate key not allowed: 'order_id'
Problem 14 duplicate-key test passed.


---

# Problem 15 — Create a Custom `JSONDecoder` Subclass

Package your decoding policy into a reusable decoder class.

Requirements:

- parse floats as `Decimal`;
- reject non-standard constants;
- reject duplicate keys;
- decode domain objects.

Then use:

```python
json.loads(text, cls=StrictDomainJSONDecoder)
```


## Solution 15


In [31]:
class StrictDomainJSONDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        # The class owns these policies, so remove conflicting caller values.
        kwargs.pop("parse_float", None)
        kwargs.pop("parse_constant", None)
        kwargs.pop("object_pairs_hook", None)
        kwargs.pop("object_hook", None)

        super().__init__(
            *args,
            parse_float=Decimal,
            parse_constant=reject_nonstandard_constant,
            object_pairs_hook=strict_domain_pairs_hook,
            **kwargs,
        )

decoded_15 = json.loads(order_json, cls=StrictDomainJSONDecoder)

assert_type(decoded_15, Order)
assert_equal(decoded_15.total, Decimal("48.40"))

print(decoded_15)
print("Problem 15 passed.")


Order(order_id='ORD-2026-0001', customer=Person(name='Grace Hopper', employee_id=7), items=(LineItem(sku='PY-BOOK', quantity=2, unit_price=Decimal('19.95')), LineItem(sku='JSON-MUG', quantity=1, unit_price=Decimal('8.50'))), created_at=datetime.datetime(2026, 8, 7, 9, 30, tzinfo=datetime.timezone.utc))
Problem 15 passed.


### Best practice

A decoder subclass is useful when a decoding policy is reused across a codebase. Keep the policy explicit and documented so callers know that ordinary JSON objects may become domain objects.


---

# Problem 16 — Decode Schema Versions and Migrate Older Payloads

Your application once encoded a person as:

### Version 1
```json
{"__type__": "person", "__version__": 1, "name": "Ada", "id": 1001}
```

The current version is:

### Version 2
```json
{
  "__type__": "person",
  "__version__": 2,
  "first_name": "Ada",
  "last_name": "Lovelace",
  "employee_id": 1001
}
```

Create a decoder that accepts both versions and returns one modern `Employee` dataclass.


## Solution 16


In [32]:
@dataclass(frozen=True)
class Employee:
    first_name: str
    last_name: str
    employee_id: int

def decode_versioned_person(obj):
    version = obj.get("__version__")

    if version == 1:
        full_name = obj["name"].strip()
        first_name, separator, last_name = full_name.partition(" ")

        if not separator:
            last_name = ""

        return Employee(
            first_name=first_name,
            last_name=last_name,
            employee_id=obj["id"],
        )

    if version == 2:
        return Employee(
            first_name=obj["first_name"],
            last_name=obj["last_name"],
            employee_id=obj["employee_id"],
        )

    raise ValueError(f"Unsupported person schema version: {version!r}")

def versioned_decoder(obj):
    if obj.get("__type__") == "person":
        return decode_versioned_person(obj)
    return obj


In [33]:
v1_json = r'''
{
  "__type__": "person",
  "__version__": 1,
  "name": "Ada Lovelace",
  "id": 1001
}
'''

v2_json = r'''
{
  "__type__": "person",
  "__version__": 2,
  "first_name": "Ada",
  "last_name": "Lovelace",
  "employee_id": 1001
}
'''

v1_employee = json.loads(v1_json, object_hook=versioned_decoder)
v2_employee = json.loads(v2_json, object_hook=versioned_decoder)

assert_equal(v1_employee, v2_employee)
print(v1_employee)
print("Problem 16 passed.")


Employee(first_name='Ada', last_name='Lovelace', employee_id=1001)
Problem 16 passed.


### Best-practice takeaway

Version your custom tagged schemas if their structure may evolve. Schema versioning is much safer than guessing a payload's meaning from whichever fields happen to be present.


---

# Problem 17 — Safe Decoding: Never Import Arbitrary Classes from JSON

A dangerous design would accept data like:

```json
{"__class__": "some.module.ClassName", "args": [...]}
```

and dynamically import and instantiate whatever the JSON requests.

Instead, build an explicit allowlist.

Support only two classes:

- `Point`
- `Rectangle`

Unknown class names must be rejected.


## Solution 17


In [34]:
@dataclass(frozen=True)
class Point:
    x: Decimal
    y: Decimal

@dataclass(frozen=True)
class Rectangle:
    top_left: Point
    width: Decimal
    height: Decimal

def decode_point(obj):
    return Point(x=obj["x"], y=obj["y"])

def decode_rectangle(obj):
    if not isinstance(obj["top_left"], Point):
        raise TypeError("Rectangle.top_left must already be a Point.")
    return Rectangle(
        top_left=obj["top_left"],
        width=obj["width"],
        height=obj["height"],
    )

SAFE_CLASS_DECODERS = {
    "Point": decode_point,
    "Rectangle": decode_rectangle,
}

def safe_class_decoder(obj):
    class_name = obj.get("__class__")
    if class_name is None:
        return obj

    decoder = SAFE_CLASS_DECODERS.get(class_name)
    if decoder is None:
        raise ValueError(f"Class is not allowed: {class_name!r}")

    return decoder(obj)


In [35]:
geometry_json = r'''
{
  "__class__": "Rectangle",
  "top_left": {
    "__class__": "Point",
    "x": 1.25,
    "y": 2.50
  },
  "width": 10.0,
  "height": 4.0
}
'''

geometry = json.loads(
    geometry_json,
    object_hook=safe_class_decoder,
    parse_float=Decimal,
)

assert_type(geometry, Rectangle)
assert_type(geometry.top_left, Point)
assert_equal(geometry.width, Decimal("10.0"))

print(geometry)


Rectangle(top_left=Point(x=Decimal('1.25'), y=Decimal('2.50')), width=Decimal('10.0'), height=Decimal('4.0'))


In [36]:
malicious_like_json = r'''
{
  "__class__": "os.system",
  "command": "echo should-never-run"
}
'''

try:
    json.loads(
        malicious_like_json,
        object_hook=safe_class_decoder,
        parse_float=Decimal,
    )
except ValueError as exc:
    print("Rejected safely:", exc)
else:
    raise AssertionError("Expected unknown class rejection.")

print("Problem 17 passed.")


Rejected safely: Class is not allowed: 'os.system'
Problem 17 passed.


### Security rule

Never let untrusted JSON choose arbitrary Python imports, constructors, functions, or executable code. Use an explicit allowlist of known decoder functions.


---

# Problem 18 — Load JSON from a File-Like Object

`json.loads` decodes from a string. `json.load` decodes from a file-like object.

Use `io.StringIO` to demonstrate `json.load` without creating an extra file.


## Solution 18


In [37]:
from io import StringIO

stream = StringIO(order_json)

decoded_18 = json.load(
    stream,
    cls=StrictDomainJSONDecoder,
)

assert_type(decoded_18, Order)
assert_equal(decoded_18.total, Decimal("48.40"))

print(decoded_18)
print("Problem 18 passed.")


Order(order_id='ORD-2026-0001', customer=Person(name='Grace Hopper', employee_id=7), items=(LineItem(sku='PY-BOOK', quantity=2, unit_price=Decimal('19.95')), LineItem(sku='JSON-MUG', quantity=1, unit_price=Decimal('8.50'))), created_at=datetime.datetime(2026, 8, 7, 9, 30, tzinfo=datetime.timezone.utc))
Problem 18 passed.


---

# Problem 19 — Round-Trip a Custom Domain Object

Custom decoding is much easier to trust when the encoder and decoder agree on a stable schema.

Write an encoder for:

- `datetime`
- `Decimal`
- `Person`
- `LineItem`
- `Order`

Then encode an `Order`, decode it, and verify equality.

> Note: JSON numeric tokens parsed with `Decimal` can preserve decimal semantics. Here we encode `Decimal` as a tagged object to make the type completely explicit.


## Solution 19


In [38]:
def encode_custom_object(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat(),
        }

    if isinstance(obj, Decimal):
        return {
            "__type__": "decimal",
            "value": str(obj),
        }

    if isinstance(obj, Person):
        return {
            "__type__": "person",
            "name": obj.name,
            "employee_id": obj.employee_id,
        }

    if isinstance(obj, LineItem):
        return {
            "__type__": "line_item",
            "sku": obj.sku,
            "quantity": obj.quantity,
            "unit_price": obj.unit_price,
        }

    if isinstance(obj, Order):
        return {
            "__type__": "order",
            "order_id": obj.order_id,
            "customer": obj.customer,
            "items": list(obj.items),
            "created_at": obj.created_at,
        }

    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


In [39]:
def decode_decimal_record(obj):
    return Decimal(obj["value"])

ROUND_TRIP_REGISTRY = {
    **DOMAIN_REGISTRY,
    "decimal": decode_decimal_record,
}

def round_trip_decoder(obj):
    tag = obj.get("__type__")
    if tag is None:
        return obj

    decoder = ROUND_TRIP_REGISTRY.get(tag)
    if decoder is None:
        raise UnknownTaggedTypeError(tag)

    return decoder(obj)


In [40]:
original_order = Order(
    order_id="ORD-RT-1",
    customer=Person("Margaret Hamilton", 42),
    items=(
        LineItem("A", 2, Decimal("12.50")),
        LineItem("B", 1, Decimal("3.75")),
    ),
    created_at=datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
)

encoded_order = json.dumps(
    original_order,
    default=encode_custom_object,
    indent=2,
)

print(encoded_order)


{
  "__type__": "order",
  "order_id": "ORD-RT-1",
  "customer": {
    "__type__": "person",
    "name": "Margaret Hamilton",
    "employee_id": 42
  },
  "items": [
    {
      "__type__": "line_item",
      "sku": "A",
      "quantity": 2,
      "unit_price": {
        "__type__": "decimal",
        "value": "12.50"
      }
    },
    {
      "__type__": "line_item",
      "sku": "B",
      "quantity": 1,
      "unit_price": {
        "__type__": "decimal",
        "value": "3.75"
      }
    }
  ],
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00+00:00"
  }
}


In [41]:
decoded_order = json.loads(
    encoded_order,
    object_hook=round_trip_decoder,
)

assert_equal(decoded_order, original_order)
assert_equal(decoded_order.total, Decimal("28.75"))

print(decoded_order)
print("Problem 19 passed.")


Order(order_id='ORD-RT-1', customer=Person(name='Margaret Hamilton', employee_id=42), items=(LineItem(sku='A', quantity=2, unit_price=Decimal('12.50')), LineItem(sku='B', quantity=1, unit_price=Decimal('3.75'))), created_at=datetime.datetime(2026, 8, 7, 12, 0, tzinfo=datetime.timezone.utc))
Problem 19 passed.


---

# Problem 20 — Build a Reusable Decoder Factory

Create a factory that accepts:

- a registry;
- `unknown="keep"` or `unknown="error"`.

It should return an `object_hook` with the requested unknown-type policy.


## Solution 20


In [42]:
def make_registry_decoder(
    registry: dict[str, Callable[[dict], Any]],
    *,
    unknown: str = "error",
):
    if unknown not in {"keep", "error"}:
        raise ValueError("unknown must be either 'keep' or 'error'")

    def decoder(obj):
        tag = obj.get("__type__")

        if tag is None:
            return obj

        decode_function = registry.get(tag)

        if decode_function is not None:
            return decode_function(obj)

        if unknown == "keep":
            return obj

        raise UnknownTaggedTypeError(f"Unknown tagged type: {tag!r}")

    return decoder

keep_unknown_decoder = make_registry_decoder(
    DECODER_REGISTRY,
    unknown="keep",
)

error_unknown_decoder = make_registry_decoder(
    DECODER_REGISTRY,
    unknown="error",
)


In [43]:
unknown_payload = r'''
{
  "x": {
    "__type__": "future_type",
    "value": 123
  }
}
'''

kept = json.loads(unknown_payload, object_hook=keep_unknown_decoder)
assert_type(kept["x"], dict)

try:
    json.loads(unknown_payload, object_hook=error_unknown_decoder)
except UnknownTaggedTypeError as exc:
    print("Strict mode rejected unknown type:", exc)
else:
    raise AssertionError("Strict mode should reject unknown type.")

print("Problem 20 passed.")


Strict mode rejected unknown type: Unknown tagged type: 'future_type'
Problem 20 passed.


---

# Problem 21 — Add Field-Level Validation to a Tagged Decoder

A fraction must satisfy:

- numerator is an integer;
- denominator is an integer;
- denominator is not zero;
- no unexpected fields are present.

Write a strict fraction decoder with clear errors.


## Solution 21


In [44]:
def decode_fraction_strict(obj):
    expected_keys = {"__type__", "numerator", "denominator"}

    if set(obj) != expected_keys:
        raise ValueError(
            f"Fraction must contain exactly {sorted(expected_keys)}; "
            f"received {sorted(obj)}"
        )

    numerator = obj["numerator"]
    denominator = obj["denominator"]

    if type(numerator) is not int:
        raise TypeError("Fraction numerator must be an integer.")

    if type(denominator) is not int:
        raise TypeError("Fraction denominator must be an integer.")

    if denominator == 0:
        raise ValueError("Fraction denominator cannot be zero.")

    return Fraction(numerator, denominator)

strict_fraction_registry = {
    "fraction": decode_fraction_strict,
}

strict_fraction_decoder = make_registry_decoder(
    strict_fraction_registry,
    unknown="error",
)


In [45]:
good_fraction = json.loads(
    '{"__type__": "fraction", "numerator": 10, "denominator": 20}',
    object_hook=strict_fraction_decoder,
)

assert_equal(good_fraction, Fraction(1, 2))

for bad_json in [
    '{"__type__": "fraction", "numerator": 1, "denominator": 0}',
    '{"__type__": "fraction", "numerator": 1.5, "denominator": 2}',
    '{"__type__": "fraction", "numerator": 1, "denominator": 2, "extra": true}',
]:
    try:
        json.loads(bad_json, object_hook=strict_fraction_decoder)
    except (TypeError, ValueError) as exc:
        print("Rejected:", exc)
    else:
        raise AssertionError(f"Expected rejection for: {bad_json}")

print("Problem 21 passed.")


Rejected: Fraction denominator cannot be zero.
Rejected: Fraction numerator must be an integer.
Rejected: Fraction must contain exactly ['__type__', 'denominator', 'numerator']; received ['__type__', 'denominator', 'extra', 'numerator']
Problem 21 passed.


---

# Problem 22 — Distinguish JSON `null` from a Rejected Constant

`null` is standard JSON and becomes Python `None`.

`NaN` is non-standard and reaches `parse_constant`.

Prove that a strict `parse_constant` does not interfere with normal `null`.


## Solution 22


In [46]:
problem_22_json = r'''
{
  "optional_value": null,
  "real_number": 1.25
}
'''

decoded_22 = json.loads(
    problem_22_json,
    parse_float=Decimal,
    parse_constant=reject_nonstandard_constant,
)

assert decoded_22["optional_value"] is None
assert_equal(decoded_22["real_number"], Decimal("1.25"))

print(decoded_22)
print("Problem 22 passed.")


{'optional_value': None, 'real_number': Decimal('1.25')}
Problem 22 passed.


---

# Problem 23 — Preserve Pair Order and Inspect Repeated Keys

Sometimes you want to inspect the raw key/value sequence instead of immediately creating a dictionary.

Use `object_pairs_hook` to return the list of pairs unchanged.


## Solution 23


In [47]:
pair_json = r'''
{
  "first": 1,
  "second": 2,
  "third": 3
}
'''

pairs = json.loads(
    pair_json,
    object_pairs_hook=lambda items: items,
)

assert_equal(
    pairs,
    [("first", 1), ("second", 2), ("third", 3)],
)

print(pairs)
print("Problem 23 passed.")


[('first', 1), ('second', 2), ('third', 3)]
Problem 23 passed.


### Modern Python note

Regular Python dictionaries preserve insertion order in current Python versions. `object_pairs_hook` is still valuable when you need access to the original pair sequence **before** dictionary construction, especially for duplicate-key detection.


---

# Problem 24 — Diagnose JSON Syntax Errors Precisely

Malformed JSON raises `json.JSONDecodeError`.

Catch the exception and report:

- message;
- line number;
- column number;
- absolute character position.


## Solution 24


In [48]:
malformed_json = r'''
{
  "name": "Python",
  "versions": [2, 3,],
  "active": true
}
'''

try:
    json.loads(malformed_json)
except json.JSONDecodeError as exc:
    print("Message :", exc.msg)
    print("Line    :", exc.lineno)
    print("Column  :", exc.colno)
    print("Position:", exc.pos)
else:
    raise AssertionError("Expected JSONDecodeError.")

print("Problem 24 passed.")


Message : Illegal trailing comma before end of array
Line    : 4
Column  : 20
Position: 42
Problem 24 passed.


### Best practice

Do not collapse all decoder errors into a generic "invalid JSON" message during development. `JSONDecodeError` gives useful location information for debugging malformed input.


---

# Problem 25 — Cap Input Size Before Decoding

`json.loads` does not provide a general "maximum document size" argument.

Write a small wrapper that refuses strings larger than a configured number of UTF-8 bytes before parsing them.


## Solution 25


In [49]:
def loads_with_size_limit(
    text: str,
    *,
    max_bytes: int,
    **json_kwargs,
):
    actual_bytes = len(text.encode("utf-8"))

    if actual_bytes > max_bytes:
        raise ValueError(
            f"JSON document too large: {actual_bytes} bytes > {max_bytes} bytes"
        )

    return json.loads(text, **json_kwargs)

small = '{"ok": true}'

decoded_25 = loads_with_size_limit(
    small,
    max_bytes=100,
)

assert_equal(decoded_25, {"ok": True})


In [50]:
try:
    loads_with_size_limit(
        '{"payload": "' + "x" * 1000 + '"}',
        max_bytes=100,
    )
except ValueError as exc:
    print("Rejected oversized document:", exc)
else:
    raise AssertionError("Expected size rejection.")

print("Problem 25 passed.")


Rejected oversized document: JSON document too large: 1015 bytes > 100 bytes
Problem 25 passed.


### Security note

Input-size limits are only one layer of defensive parsing. Real systems may also need transport limits, nesting limits, schema validation, timeouts, and resource controls.


---

# Problem 26 — One Production-Style Strict Loader

Combine several best practices into one reusable function:

- document-size limit;
- `Decimal` for JSON floats;
- reject `NaN` / infinities;
- reject duplicate keys;
- decode only allowlisted tagged types;
- reject unknown tagged types.

Use the small registry from Problem 10.


## Solution 26


In [51]:
PRODUCTION_REGISTRY = {
    "datetime": decode_datetime_record,
    "date": decode_date_record,
    "fraction": decode_fraction_strict,
    "uuid": decode_uuid_record,
}

production_object_decoder = make_registry_decoder(
    PRODUCTION_REGISTRY,
    unknown="error",
)

def production_pairs_hook(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate key: {key!r}")
        obj[key] = value

    return production_object_decoder(obj)

def strict_loads(text: str, *, max_bytes: int = 1_000_000):
    return loads_with_size_limit(
        text,
        max_bytes=max_bytes,
        parse_float=Decimal,
        parse_constant=reject_nonstandard_constant,
        object_pairs_hook=production_pairs_hook,
    )


In [52]:
production_json = r'''
{
  "request_id": {
    "__type__": "uuid",
    "value": "12345678-1234-5678-1234-567812345678"
  },
  "created": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00Z"
  },
  "ratio": {
    "__type__": "fraction",
    "numerator": 2,
    "denominator": 5
  },
  "amount": 12.99,
  "optional": null
}
'''

decoded_26 = strict_loads(production_json)

assert_type(decoded_26["request_id"], UUID)
assert_type(decoded_26["created"], datetime)
assert_equal(decoded_26["ratio"], Fraction(2, 5))
assert_equal(decoded_26["amount"], Decimal("12.99"))
assert decoded_26["optional"] is None

decoded_26


{'request_id': UUID('12345678-1234-5678-1234-567812345678'),
 'created': datetime.datetime(2026, 8, 7, 12, 0, tzinfo=datetime.timezone.utc),
 'ratio': Fraction(2, 5),
 'amount': Decimal('12.99'),
 'optional': None}

In [53]:
print("Problem 26 passed.")


Problem 26 passed.


---

# Problem 27 — Mini Challenge: Event Stream Decoder

Build an event-stream decoder using these schemas:

```json
{
  "__type__": "event",
  "event_id": {"__type__": "uuid", "value": "..."},
  "occurred_at": {"__type__": "datetime", "value": "..."},
  "kind": "purchase",
  "amount": 49.95
}
```

Requirements:

- `event_id` -> `UUID`
- `occurred_at` -> timezone-aware `datetime`
- `amount` -> `Decimal`
- outer object -> `Event`
- unknown tagged types -> error
- duplicate keys -> error
- non-standard constants -> error


## Solution 27


In [54]:
@dataclass(frozen=True)
class Event:
    event_id: UUID
    occurred_at: datetime
    kind: str
    amount: Decimal

def decode_event_record(obj):
    event_id = obj["event_id"]
    occurred_at = obj["occurred_at"]
    kind = obj["kind"]
    amount = obj["amount"]

    if not isinstance(event_id, UUID):
        raise TypeError("event_id must decode to UUID.")

    if not isinstance(occurred_at, datetime):
        raise TypeError("occurred_at must decode to datetime.")

    if occurred_at.tzinfo is None:
        raise ValueError("occurred_at must be timezone-aware.")

    if not isinstance(kind, str) or not kind:
        raise ValueError("kind must be a non-empty string.")

    if not isinstance(amount, Decimal):
        raise TypeError("amount must be Decimal.")

    return Event(
        event_id=event_id,
        occurred_at=occurred_at,
        kind=kind,
        amount=amount,
    )

EVENT_REGISTRY = {
    "uuid": decode_uuid_record,
    "datetime": decode_datetime_record,
    "event": decode_event_record,
}

event_object_decoder = make_registry_decoder(
    EVENT_REGISTRY,
    unknown="error",
)

def event_pairs_hook(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate key: {key!r}")
        obj[key] = value

    return event_object_decoder(obj)


In [55]:
event_json = r'''
{
  "__type__": "event",
  "event_id": {
    "__type__": "uuid",
    "value": "aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee"
  },
  "occurred_at": {
    "__type__": "datetime",
    "value": "2026-08-07T15:45:00+03:00"
  },
  "kind": "purchase",
  "amount": 49.95
}
'''

event = json.loads(
    event_json,
    object_pairs_hook=event_pairs_hook,
    parse_float=Decimal,
    parse_constant=reject_nonstandard_constant,
)

event


Event(event_id=UUID('aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee'), occurred_at=datetime.datetime(2026, 8, 7, 15, 45, tzinfo=datetime.timezone(datetime.timedelta(seconds=10800))), kind='purchase', amount=Decimal('49.95'))

In [56]:
assert_type(event, Event)
assert_type(event.event_id, UUID)
assert_type(event.occurred_at, datetime)
assert_type(event.amount, Decimal)
assert_equal(event.amount, Decimal("49.95"))

print("Problem 27 passed.")


Problem 27 passed.


---

# Problem 28 — Batch Decode and Validate Many Events

Decode an array of events. The top-level JSON value is an array, so `object_hook`/`object_pairs_hook` is applied to each object inside it, but not to the list itself.

Compute the total amount afterward.


## Solution 28


In [57]:
events_json = r'''
[
  {
    "__type__": "event",
    "event_id": {
      "__type__": "uuid",
      "value": "00000000-0000-0000-0000-000000000001"
    },
    "occurred_at": {
      "__type__": "datetime",
      "value": "2026-08-07T10:00:00Z"
    },
    "kind": "purchase",
    "amount": 10.10
  },
  {
    "__type__": "event",
    "event_id": {
      "__type__": "uuid",
      "value": "00000000-0000-0000-0000-000000000002"
    },
    "occurred_at": {
      "__type__": "datetime",
      "value": "2026-08-07T11:00:00Z"
    },
    "kind": "purchase",
    "amount": 20.20
  },
  {
    "__type__": "event",
    "event_id": {
      "__type__": "uuid",
      "value": "00000000-0000-0000-0000-000000000003"
    },
    "occurred_at": {
      "__type__": "datetime",
      "value": "2026-08-07T12:00:00Z"
    },
    "kind": "purchase",
    "amount": 30.30
  }
]
'''

events = json.loads(
    events_json,
    object_pairs_hook=event_pairs_hook,
    parse_float=Decimal,
    parse_constant=reject_nonstandard_constant,
)

assert_type(events, list)
assert all(isinstance(item, Event) for item in events)

grand_total = sum((item.amount for item in events), Decimal("0"))
assert_equal(grand_total, Decimal("60.60"))

print("Grand total:", grand_total)
print("Problem 28 passed.")


Grand total: 60.60
Problem 28 passed.


---

# Problem 29 — Compare `parse_float=Decimal` with Default Float Parsing

Decode the same JSON twice and compare types and arithmetic.


## Solution 29


In [58]:
numeric_json = '{"a": 0.1, "b": 0.2}'

with_float = json.loads(numeric_json)
with_decimal = json.loads(numeric_json, parse_float=Decimal)

print("float types   :", type(with_float["a"]), type(with_float["b"]))
print("Decimal types :", type(with_decimal["a"]), type(with_decimal["b"]))

print("float sum     :", with_float["a"] + with_float["b"])
print("Decimal sum   :", with_decimal["a"] + with_decimal["b"])


float types   : <class 'float'> <class 'float'>
Decimal types : <class 'decimal.Decimal'> <class 'decimal.Decimal'>
float sum     : 0.30000000000000004
Decimal sum   : 0.3


In [59]:
assert with_float["a"] + with_float["b"] != 0.3
assert_equal(with_decimal["a"] + with_decimal["b"], Decimal("0.3"))

print("Problem 29 passed.")


Problem 29 passed.


---

# Problem 30 — Final Integrated Challenge

Write a strict configuration decoder.

Supported tagged values:

- `datetime`
- `date`
- `uuid`
- `fraction`
- `decimal`

Rules:

1. Reject duplicate keys.
2. Reject `NaN`, `Infinity`, `-Infinity`.
3. Parse ordinary floating-point numbers as `Decimal`.
4. Reject unknown tagged types.
5. Reject documents over 50 KiB.
6. Return ordinary untagged objects as dictionaries.

Then decode a realistic application configuration.


## Solution 30


In [60]:
def decode_decimal_tag(obj):
    value = obj["value"]
    if not isinstance(value, str):
        raise TypeError("Tagged decimal value must be stored as a string.")
    return Decimal(value)

CONFIG_REGISTRY = {
    "datetime": decode_datetime_record,
    "date": decode_date_record,
    "uuid": decode_uuid_record,
    "fraction": decode_fraction_strict,
    "decimal": decode_decimal_tag,
}

config_object_decoder = make_registry_decoder(
    CONFIG_REGISTRY,
    unknown="error",
)

def config_pairs_hook(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"Duplicate configuration key: {key!r}")
        obj[key] = value

    return config_object_decoder(obj)

def load_config(text: str):
    return loads_with_size_limit(
        text,
        max_bytes=50 * 1024,
        parse_float=Decimal,
        parse_constant=reject_nonstandard_constant,
        object_pairs_hook=config_pairs_hook,
    )


In [61]:
config_json = r'''
{
  "service": {
    "name": "billing",
    "enabled": true,
    "retry_backoff": 0.125,
    "launched": {
      "__type__": "date",
      "value": "2026-08-01"
    }
  },
  "instance_id": {
    "__type__": "uuid",
    "value": "f47ac10b-58cc-4372-a567-0e02b2c3d479"
  },
  "maintenance": {
    "starts": {
      "__type__": "datetime",
      "value": "2026-08-09T01:00:00+03:00"
    },
    "traffic_fraction": {
      "__type__": "fraction",
      "numerator": 1,
      "denominator": 4
    }
  },
  "limits": {
    "monthly_budget": {
      "__type__": "decimal",
      "value": "10000.00"
    },
    "warning_ratio": 0.85
  }
}
'''

config = load_config(config_json)
config


{'service': {'name': 'billing',
  'enabled': True,
  'retry_backoff': Decimal('0.125'),
  'launched': datetime.date(2026, 8, 1)},
 'instance_id': UUID('f47ac10b-58cc-4372-a567-0e02b2c3d479'),
 'maintenance': {'starts': datetime.datetime(2026, 8, 9, 1, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=10800))),
  'traffic_fraction': Fraction(1, 4)},
 'limits': {'monthly_budget': Decimal('10000.00'),
  'warning_ratio': Decimal('0.85')}}

In [62]:
assert_equal(config["service"]["name"], "billing")
assert config["service"]["enabled"] is True
assert_equal(config["service"]["retry_backoff"], Decimal("0.125"))
assert_type(config["service"]["launched"], date)
assert_type(config["instance_id"], UUID)
assert_type(config["maintenance"]["starts"], datetime)
assert_equal(config["maintenance"]["traffic_fraction"], Fraction(1, 4))
assert_equal(config["limits"]["monthly_budget"], Decimal("10000.00"))
assert_equal(config["limits"]["warning_ratio"], Decimal("0.85"))

print("Problem 30 passed.")


Problem 30 passed.


---

# Best-Practice Checklist

Use this checklist when designing custom JSON decoding in production:

1. **Use explicit schemas/tags.** Do not guess object types from vague field combinations.
2. **Reserve tag names.** Names such as `__type__` and `__version__` reduce accidental collisions.
3. **Validate fields before construction.** Check required keys, unexpected keys, and value types.
4. **Use allowlists.** Never let untrusted JSON choose arbitrary Python imports/classes/functions.
5. **Use `Decimal` for exact decimal-domain values** when binary floating-point error is unacceptable.
6. **Reject non-standard constants** if strict JSON is required.
7. **Detect duplicate keys** when ambiguity could be dangerous.
8. **Version evolving custom schemas.**
9. **Exploit bottom-up hooks** for nested object reconstruction.
10. **Keep decoder functions small and testable.**
11. **Prefer a registry over large `if/elif` chains** once the number of tagged types grows.
12. **Apply resource limits** before or around parsing untrusted input.
13. **Test malformed and adversarial inputs**, not only happy paths.
14. **Round-trip test** encoder/decoder pairs when both are under your control.
15. **Document whether unknown tags are preserved or rejected.**

## Core API summary

```python
json.loads(
    text,
    object_hook=...,
    object_pairs_hook=...,
    parse_float=...,
    parse_int=...,
    parse_constant=...,
    cls=...,
)
```

Remember:

- `object_hook` receives each decoded **dictionary**.
- `object_pairs_hook` receives an ordered **list of `(key, value)` pairs**.
- If both are supplied, `object_pairs_hook` takes precedence.
- `parse_float`, `parse_int`, and `parse_constant` receive textual numeric/constant tokens.
- Arrays decode to Python lists unless you transform them later.


# Optional Self-Test Ideas

Try extending the notebook yourself:

- Add support for `complex`, `set`, and `Path`.
- Add a `Money(currency, amount)` dataclass with currency validation.
- Reject naive datetimes and require offsets.
- Add a schema version for `Order`.
- Decode an enum from a tagged object.
- Build an encoder/decoder pair for `Rectangle`.
- Add unit tests for every invalid payload.
- Compare `object_hook` with a post-processing recursive walk.
- Add maximum list-length validation after parsing.
- Add application-specific error classes with structured error codes.
